In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import random
import numpy as np
from sklearn.metrics import f1_score
from datetime import datetime


def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


set_random_seed(42)

In [ ]:
# --- 1. Настройка окружения и девайса ---
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Используем GPU для ускорения.")
elif torch.backends.mps.is_available():
    device = torch.device("mps") # Для Mac M1/M2
    print("Используем MPS (Mac) для ускорения.")
else:
    print("Используем CPU. Это может быть медленно.")


In [ ]:
# ==========================================
# 2. Функция вычисления метрик
# ==========================================

def evaluate_metrics(model, test_loader, loss_fn, device):
    """
    Вычисляет Accuracy, F1-score и Optimality Gap.
    """
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    # Списки для сбора всех предсказаний и истинных меток (нужны для F1)
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            # 1. Считаем Loss (Это наш прокси для Optimality Gap)
            loss = loss_fn(output, target)
            total_loss += loss.item() * data.size(0)

            # 2. Считаем Accuracy
            _, pred = torch.max(output.data, 1)
            correct += (pred == target).sum().item()
            total += data.size(0)

            # Сохраняем данные для F1-Score
            all_targets.extend(target.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    # Итоговые метрики
    accuracy = 100.0 * correct / total
    avg_loss = total_loss / total - 0.0333  # Opt. Gap min: 0.0333

    # Вычисляем Macro F1-Score
    macro_f1 = f1_score(all_targets, all_preds, average='macro') * 100.0

    return accuracy, macro_f1, avg_loss

In [ ]:
# ==========================================
# 3. АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (LeNet-5 для MNIST)
# ==========================================
class MNIST_LeNet(nn.Module):
    def __init__(self):
        super(MNIST_LeNet, self).__init__()
        # Вход: 1 канал (серый), выход: 6 фильтров, ядро 5x5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2) # Padding=2 сохраняет 28x28
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # После conv + pool размер станет 5x5
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
# ==========================================
# 4. КЛИЕНТСКАЯ ЧАСТЬ (Модуль SAM)
# ==========================================
def compute_sam_gradient(model, loss_fn, data, target, rho=0.05):
    """
    Вычисляет робастный градиент с помощью Sharpness-Aware Minimization.
    """
    # Сохраняем исходные веса w^t, чтобы вернуться к ним после "прыжка"
    original_weights = {name: param.clone() for name, param in model.named_parameters()}

    # --- Шаг 1: Вычисляем обычный градиент g = \nabla F(w^t) ---
    model.zero_grad()
    predictions = model(data)
    loss = loss_fn(predictions, target)
    loss.backward()

    # Вычисляем общую L2-норму градиентов со всех слоев модели
    grad_norm = torch.norm(
        torch.stack([param.grad.norm(p=2) for param in model.parameters() if param.grad is not None]),
        p=2
    )

    # --- Шаг 2: Виртуальный прыжок в худшую точку (w^t + \epsilon) ---
    if grad_norm > 0:
        # scale = \rho / ||g||_2  (добавляем 1e-12 для защиты от деления на ноль)
        scale = rho / (grad_norm + 1e-12)
        for param in model.parameters():
            if param.grad is not None:
                e_w = param.grad * scale
                param.data.add_(e_w) # Физически сдвигаем веса в точку w + \epsilon

    # --- Шаг 3: Вычисление SAM-градиента в этой "плохой" точке ---
    model.zero_grad() # Очищаем старые градиенты
    predictions_adv = model(data)
    loss_adv = loss_fn(predictions_adv, target)
    loss_adv.backward() # Теперь в param.grad лежит g^{SAM}

    # Собираем итоговые SAM-градиенты в список для отправки на сервер
    sam_gradients = []
    for param in model.parameters():
        if param.grad is not None:
            sam_gradients.append(param.grad.clone())

    # ВАЖНО: Возвращаем веса модели обратно в исходную точку w^t
    for name, param in model.named_parameters():
        param.data.copy_(original_weights[name])

    return sam_gradients

In [ ]:
# ==========================================
# 5. СЕРВЕРНАЯ ЧАСТЬ (L2-Нормализация и Агрегация)
# ==========================================
def server_aggregate_and_update(global_model, client_gradients_list, eta_glob=1.0):
    """
    Собирает градиенты, нормализует их и обновляет глобальную модель.
    client_gradients_list: список списков градиентов от каждого клиента.
    """
    # Инициализируем пустой аккумулятор для итогового вектора
    aggregated_grad = [torch.zeros_like(param) for param in global_model.parameters()]
    num_clients = len(client_gradients_list)

    # Обрабатываем градиент каждого клиента по очереди
    for client_grads in client_gradients_list:

        # Вычисляем L2-норму вектора, присланного клиентом (даже если он хакер)
        client_norm = torch.norm(
            torch.stack([g.norm(p=2) for g in client_grads]),
            p=2
        )

        # L2-Нормализация и усреднение (добавляем вклад клиента)
        for i, g in enumerate(client_grads):
            # Делаем длину вектора равной 1
            normalized_g = g / (client_norm + 1e-12)
            # Добавляем в общую копилку (с весом 1/M)
            aggregated_grad[i] += normalized_g / num_clients

    # --- Шаг обновления глобальной модели (w^{t+1} = w^t - \eta_{glob} * g_{agg}) ---
    with torch.no_grad():
        for param, avg_grad in zip(global_model.parameters(), aggregated_grad):
            param.data.sub_(eta_glob * avg_grad)

In [ ]:
# ==========================================
# 6. Функция распределения Дирихле (Non-IID)
# ==========================================
def partition_data_dirichlet(dataset, num_clients, beta=0.5, num_classes=10):
    """
    Разделяет набор данных между клиентами используя распределение Дирихле.

    Параметры:
        dataset: PyTorch Dataset
        num_clients: Количество клиентов
        beta: Параметр концентрации Дирихле (меньше = больше перекос/Non-IID)
        num_classes: Количество классов в датасете

    Возвращает:
        Список объектов Subset, где каждый Subset содержит данные одного клиента.
    """
    # Извлекаем все метки классов из датасета
    # Для torchvision.datasets (как MNIST) метки хранятся в атрибуте targets
    if hasattr(dataset, 'targets'):
        train_labels = np.array(dataset.targets)
    else:
        # Универсальный (но медленный) способ, если атрибут targets недоступен
        train_labels = np.array([target for _, target in dataset])

    min_size = 0
    min_require_size = 10 # Гарантируем, что каждый клиент получит хоть какие-то данные

    # Массив для хранения индексов данных каждого клиента
    client_indices = [[] for _ in range(num_clients)]

    # Мы повторяем распределение, пока не убедимся, что ни один клиент не остался без данных
    while min_size < min_require_size:
        client_indices = [[] for _ in range(num_clients)]

        for k in range(num_classes):
            # Находим все индексы изображений класса k
            idx_k = np.where(train_labels == k)[0]
            np.random.shuffle(idx_k)

            # Генерируем пропорции распределения этого класса между клиентами
            # Математика: p ~ Dir(beta * 1)
            proportions = np.random.dirichlet(np.repeat(beta, num_clients))

            # Чтобы избежать микро-долей, отсекаем слишком маленькие пропорции
            # и нормализуем вектор, чтобы сумма снова равнялась 1
            proportions = np.array([p * (len(idx_j) < len(train_labels) / num_clients)
                                    for p, idx_j in zip(proportions, client_indices)])
            proportions = proportions / proportions.sum()

            # Превращаем пропорции в конкретное количество образцов для каждого клиента
            proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]

            # Разделяем индексы класса k между клиентами согласно пропорциям
            idx_k_split = np.split(idx_k, proportions)

            for i in range(num_clients):
                client_indices[i] += idx_k_split[i].tolist()

        # Проверяем, сколько данных получил самый "бедный" клиент
        min_size = min([len(idx_j) for idx_j in client_indices])

    # Превращаем списки индексов в объекты PyTorch Subset
    client_datasets = [Subset(dataset, indices) for indices in client_indices]

    # Небольшой вывод статистики для наглядности
    print(f"--- Распределение Дирихле (beta={beta}) завершено ---")
    for i, indices in enumerate(client_indices):
        print(f"Клиент {i+1}: {len(indices)} образцов данных")

    return client_datasets

In [ ]:
# ================================
# 7. В ЭТОМ БЛОКЕ РЕАЛИЗУЮТСЯ АТАКИ
# ================================

# ================================
# 7.1 Атаки на градиенты
# ================================
def apply_gradient_attacks(client_gradients, num_attackers, attack_type="lie", z_val=1.5, c_val=100.0, gamma=5.0):
    """
    Применяет византийские атаки к первым `num_attackers` клиентам. Учитывая случайность распределения
    данных между клиентами, это не нарушает обобщенность атаки.
    client_gradients: список градиентов от всех клиентов.
    """
    if num_attackers == 0:
        return client_gradients
        
    num_clients = len(client_gradients)
    honest_gradients = client_gradients[num_attackers:]
    attacked_gradients = client_gradients.copy()
    
    # Собираем честные градиенты для вычисления статистики (нужно для умных атак)
    # Формат: список тензоров для каждого слоя
    honest_stacked = [torch.stack([h[layer_idx] for h in honest_gradients]) 
                      for layer_idx in range(len(honest_gradients[0]))]

    for i in range(num_attackers):
        malicious_grad = []
        for layer_idx in range(len(client_gradients[0])):
            honest_layer_stack = honest_stacked[layer_idx]

            # Gaussian attack (Fed-NGA)
            if attack_type == "gaussian":
                # Генерируем шум с теми же средним и дисперсией, что у честных
                mean = torch.mean(honest_layer_stack, dim=0)
                std = torch.std(honest_layer_stack, dim=0) + 1e-6
                noise = torch.randn_like(mean) * std + mean
                malicious_grad.append(noise)
                
            # Same-value attack (Fed-NGA)
            elif attack_type == "same_value":
                # Заполняем весь тензор константой (по умолчанию 100.0)
                malicious_grad.append(torch.full_like(honest_layer_stack[0], c_val))
                
            # Double Attack (FedLAW)
            elif attack_type == "double":
                # Умножаем честный градиент самого хакера на -2 (Sign-Flipping x2)
                malicious_grad.append(-gamma * client_gradients[i][layer_idx])
                
            # Lie Attack / ALIE - A Little Is Enough (FedLAW)
            elif attack_type == "lie":
                # A Little Is Enough (ALIE)
                mean = torch.mean(honest_layer_stack, dim=0)
                std = torch.std(honest_layer_stack, dim=0) + 1e-6
                # Смещаемся от среднего на z_val стандартных отклонений
                stealth_vector = mean - z_val * std
                malicious_grad.append(stealth_vector)
                
        attacked_gradients[i] = malicious_grad
        
    return attacked_gradients

In [ ]:
# ==========================================
# 8. ОСНОВНОЙ ЦИКЛ СИМУЛЯЦИИ
# ==========================================
# Модель LeNet-5 для датасета MNIST
# ==========================================

print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 10
rounds = 400
batch_size = 64
num_attackers = 1  # <= 40% число атакующих из num_clients
attack_type = "lie" # "gaussian", "same_value", "double", "lie"
eta_glob = 0.5

# === Non-IID Распределение данных === 
beta_value = 0.1 # Изменить на 100 для IID, или на 0.1 для экстремального перекоса
set_random_seed(42)
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)

loss_fn = nn.CrossEntropyLoss()


In [ ]:
if attack_type == "gaussian":
    print("Атака: Gaussian attack")
elif attack_type == "same_value":
    print("Атака: Same-value attack")
elif attack_type == "double":
    print("Атака: Double Attack")
elif attack_type == "lie":
    print("Атака: Lie Attack / ALIE - A Little Is Enough")

print(f"Количество атакующих: {num_attackers}\n==========================================================================================")
print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")
start_time = datetime.now()

# Инициируем модель
global_model = MNIST_LeNet().to(device)

accuracy_list, f1_list, optimality_gap_list = [], [], []  # Создаем списки для метрик

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Создаем пустые тензоры для накопления градиентов за всю эпоху
        client_accumulated_grads = [torch.zeros_like(param) for param in global_model.parameters()]
        num_batches = 0

        # === Проходим по ВСЕМ батчам клиента (Локальная Эпоха) ===
        for data, target in client_loaders[client_id]:
            data, target = data.to(device), target.to(device)

            # Вычисляем SAM-градиент для текущего батча (64 картинки)
            batch_grads = compute_sam_gradient(global_model, loss_fn, data, target, rho=0.05)

            # Прибавляем градиенты батча к общей сумме
            for i, g in enumerate(batch_grads):
                client_accumulated_grads[i] += g

            num_batches += 1

        # Усредняем градиенты за всю эпоху
        client_avg_grads = [g / num_batches for g in client_accumulated_grads]
        client_gradients.append(client_avg_grads)

    # === Внедрение византийской атаки ===
    client_gradients = apply_gradient_attacks(client_gradients, num_attackers, attack_type=attack_type)

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты и делает шаг
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)
    accuracy_list.append(accuracy)
    f1_list.append(f1)
    optimality_gap_list.append(optimality_gap)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {str(datetime.now()-start_time).split('.')[0]}")

    global_model.train() # Возвращаем модель в режим обучения

print(f"Accuracy max: {max(accuracy_list):.2f}% | F1-Score max: {max(f1_list):.2f}% | Opt. Gap min: {min(optimality_gap_list):.4f}")
